# FMGAI Group Assignment, Phase 2: noise-schedule ablation on InstructPix2Pix

4× aerial super-resolution on UC Merced. Run 0 = pretrained, Run A = original schedule {β_t}, Run B = {β_t^x}, x = 1.60.

**Setup:** upload the `phase2` folder to Google Drive at `MyDrive/FMGAI_A1/phase2`, then choose *Runtime → Change runtime type → GPU*.
Results go to `MyDrive/FMGAI_A1/results`. Each training run takes about 1.5–3 h on a T4; keep the tab open. If the session dies, rerun the setup
cells (1–3) and then the same training cell. Finished steps are on Drive and never need rerunning.

The same steps run on the IITJ HPC (Slurm) through `hpc/submit.sh`; see the README.

In [ ]:
# 1. Mount Drive and go to the code folder
from google.colab import drive
drive.mount('/content/drive')
PROJ = '/content/drive/MyDrive/FMGAI_A1/phase2'
OUT = '/content/drive/MyDrive/FMGAI_A1/results'
DATA = '/content/data'          # rebuilt locally each session (deterministic, about 2 min); faster than reading from Drive
import os
os.makedirs(OUT, exist_ok=True)
%cd {PROJ}
!nvidia-smi

In [ ]:
# 2. Install and record the exact environment for the README
!pip install -q -r requirements.txt
!python env_report.py | tee {OUT}/environment.txt

In [ ]:
# 3. Build the LR/HR pairs and the 1470/315/315 split
!python prepare_data.py --data_dir {DATA}
!cp {DATA}/splits.csv {DATA}/dataset_summary.json {OUT}/

In [ ]:
# 4. Schedule Analysis figures and the original-vs-modified code listing
!python plot_schedule.py --out_dir {OUT}/figures
!python make_diff.py --out_dir {OUT}/figures

### Optional smoke test (about 5 min)
One short epoch on 16 images to check the whole loop (CSV, validation, samples, saving) before the long runs. It writes to `/content/smoke` only.

In [ ]:
!rm -rf /content/smoke
!bash run_train.sh power /content/smoke {DATA} --max_train_samples 16 --num_train_epochs 1 --num_val_metric_images 8
!cat /content/smoke/train_log.csv /content/smoke/val_metrics.csv /content/smoke/schedule_check.json && ls /content/smoke/samples

In [ ]:
# 5. Run 0 (pretrained, no fine-tuning) and the bicubic reference on the test split
!python infer.py --run bicubic --data_dir {DATA} --out_dir {OUT}/eval
!python infer.py --run run0 --data_dir {DATA} --out_dir {OUT}/eval

In [ ]:
# 6. The fine-tuning arguments are in run_train.sh (shared with the HPC jobs), identical for Run A and Run B
!sed -n '/^exec/,$p' run_train.sh

In [ ]:
# 7. Run A: original schedule {beta_t}
# The 5 GB training checkpoints stay on Colab's local disk (Drive's 15 GB would fill up, and files deleted from
# Drive go to its Trash). When training finishes, logs, samples and the final fp16 model (2.1 GB) are copied to Drive.
# If the session dies mid-run, rerun cells 1-3 and this cell: the run starts again from epoch 1.
!bash run_train.sh original /content/outputs/runA {DATA} && mkdir -p {OUT}/outputs && rsync -a --exclude 'checkpoint-*' /content/outputs/runA {OUT}/outputs/
!ls {OUT}/outputs/runA && du -sh {OUT}/outputs/runA

In [ ]:
# 8. Run B: modified schedule {beta_t ** 1.60}
# The 5 GB training checkpoints stay on Colab's local disk (Drive's 15 GB would fill up, and files deleted from
# Drive go to its Trash). When training finishes, logs, samples and the final fp16 model (2.1 GB) are copied to Drive.
# If the session dies mid-run, rerun cells 1-3 and this cell: the run starts again from epoch 1.
!bash run_train.sh power /content/outputs/runB {DATA} && mkdir -p {OUT}/outputs && rsync -a --exclude 'checkpoint-*' /content/outputs/runB {OUT}/outputs/
!ls {OUT}/outputs/runB && du -sh {OUT}/outputs/runB

In [ ]:
# 9. Test-set scores and inference trajectories of the final models
!python infer.py --run runA --data_dir {DATA} --out_dir {OUT}/eval --model_dir {OUT}/outputs/runA
!python infer.py --run runB --data_dir {DATA} --out_dir {OUT}/eval --model_dir {OUT}/outputs/runB

In [ ]:
# 10. Report figures and the results table
!python make_figures.py --runs_dir {OUT}/outputs --eval_dir {OUT}/eval --data_dir {DATA} --out_dir {OUT}/figures
from IPython.display import Image, display, Markdown
display(Markdown(open(f'{OUT}/figures/results_table.md').read()))
for f in ['loss_curves.png', 'checkpoints_runA.png', 'checkpoints_runB.png', 'test_comparison.png']:
    display(Image(f'{OUT}/figures/{f}'))
display(Image(f'{OUT}/eval/runA/trajectories/runA_trajectory.png'))
display(Image(f'{OUT}/eval/runB/trajectories/runB_trajectory.png'))

In [ ]:
# 11. Submission zip: code, logs, samples, trajectories, figures (no model weights). Add the report PDF before submitting.
!python make_submission.py --results_dir {OUT}